# Learned lead scoring vs rules — EDA & results

**JMLC research exhibit.** Can a model trained on (LLM-generated) calls rank
*who to call back first* better than the hand-written rule scorer — and **how do
we know the signal is real, not a generator artifact?**

> ## ⚠️ DATA MODE
> This notebook auto-detects its data. If `data/raw/{train,test,ood}.jsonl` exist
> (produced by `make data` with a real `GEN_MODEL` + API key), it uses **those**.
> Otherwise it falls back to an **OFFLINE-SMOKE** dataset: a small, deterministic
> *Python stub* role-plays each lead from the behavior brief — **no LLM, no
> network, no real calls.** Offline numbers are **illustrative only**; they exist
> so the full pipeline (features → model → leakage guard → OOD → calibration)
> executes end-to-end and `make report` renders. They are **not** a result.
>
> **Integrity:** synthetic is always labeled synthetic; the OFFLINE-SMOKE stub is
> doubly-synthetic (not even LLM). The real-call set is a *separate*, genuinely
> collected, honestly-sized layer — absent here until collected via the prod app.

In [ ]:
import json, warnings, os, sys
from pathlib import Path
# Bootstrap: make the repo root importable + the working dir, no matter where this
# notebook is executed from (nbconvert runs the kernel in notebooks/).
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "ml" / "config.py").exists():
        os.chdir(_p); sys.path.insert(0, str(_p)); break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from ml.config import load_scenarios, scenarios_by_id, RAW
from ml.generate import generate_dataset
from ml.train import train_model
from ml.evaluate import baseline_scores, model_scores, compare, leakage_auc, _read_jsonl
from ml.metrics import evaluate_scores
from ml.features import structured_features

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
RNG_SEED = 42
SCS_BY_ID = scenarios_by_id(load_scenarios())
print("scenarios:", list(SCS_BY_ID))

## 1. Data — real if present, else OFFLINE-SMOKE

The OFFLINE-SMOKE generator below is **scaffolding, not the method**. It reads the
same behavior brief the real LLM would see (band-level trait cues, no numbers, no
label) and emits a short transcript + captured fields. Two *surfaces* are used:
`gen-A` (train/test) and `gen-B` (OOD) — same latent→label process, different
wording — so the distribution-shift test is meaningful. Labels still come from the
real `personas.sample_label` (continuous θ + noise), never from the stub.

In [ ]:
# ===========================================================================
# OFFLINE-SMOKE stub.  *** NOT an LLM.  NOT real calls. ***  Deterministic.
# Replace by running `make data` with a real GEN_MODEL + key for the real study.
# ===========================================================================
ENG = SCS_BY_ID["english-courses-qualify"]

def _band_of(user, hi, lo):
    return 2 if hi in user else (0 if lo in user else 1)

def make_offline_complete_fn(seed=0, style="A"):
    rng = np.random.default_rng(seed)
    # gen-A vocab. readiness & urgency are the dominant label drivers -> richest phrasing.
    A = {
        "readiness": {0: ["пока просто смотрю что есть", "ещё совсем не решил", "интересуюсь без обязательств"],
                      1: ["думаю над этим", "присматриваюсь к занятиям"],
                      2: ["я уже решил заниматься", "хочу записаться на курс", "точно буду учиться"]},
        "urgency": {0: ["время терпит совсем", "может потом когда-нибудь", "не горит вообще"],
                    1: ["хотел бы в этом месяце"],
                    2: ["хочу начать на этой неделе", "это срочно для меня", "стартовать бы поскорее"]},
        "budget": {0: ["переживаю за цену", "есть ли рассрочка"],
                   1: ["сколько это стоит"],
                   2: ["по цене вопросов нет"]},
        "program": {0: ["не уверен что мне подходит"],
                    1: ["в целом подходит мне"],
                    2: ["это именно то что нужно"]},
        "engage": {0: ["да", "нет", "ну не знаю"],
                   1: ["в общем да, понятно"],
                   2: ["да, конечно, расскажу подробнее, и сам хочу спросить пару вещей"]},
    }
    # gen-B (OOD) alternates. gen-B reuses A's vocab AND adds these (superset)
    # -> lexical overlap, so an A-trained model transfers and OOD degrades gracefully.
    B = {
        "readiness": {0: ["я только изучаю варианты", "конкретики пока нет", "без конкретных планов"],
                      1: ["рассматриваю возможность", "взвешиваю"],
                      2: ["настроен учиться", "давайте оформлять", "решение принято, буду учиться"]},
        "urgency": {0: ["спешки никакой", "когда-нибудь в будущем"],
                    1: ["планирую в течение месяца"],
                    2: ["стартовать бы прямо сейчас", "очень нужно быстро"]},
        "budget": {0: ["цена смущает", "хотелось бы скидку"], 1: ["какой ценник"], 2: ["со стоимостью всё ок"]},
        "program": {0: ["мой случай чуть другой"], 1: ["мне это в принципе годится"], 2: ["прямо моя тема"]},
        "engage": {0: ["ага", "не", "хз"], 1: ["да, в целом ясно"],
                   2: ["конечно, опишу детально, и вопросы есть с моей стороны"]},
    }
    if style == "A":
        FR, FILLER = A, ["эм", "ну", "в общем"]
    else:
        FR = {tr: {b: list(A[tr][b]) + list(B[tr][b]) for b in (0, 1, 2)} for tr in A}
        FILLER = ["эм", "ну", "короче", "типа"]

    def pick(bmap, b): return rng.choice(bmap[b])
    def maybe(p): return rng.random() < p

    def complete(system, user, temperature=0.9):
        r = _band_of(user, "решил", "просто собираешь информацию")
        u = _band_of(user, "как можно скорее", "не торопишься")
        b = _band_of(user, "не проблема", "бюджет ограничен")
        p = _band_of(user, "точно совпадает", "лишь частично")
        e = _band_of(user, "развёрнуто", "односложно")
        # dominant drivers always stated distinctly -> clean text signal
        frags = [pick(FR["readiness"], r), pick(FR["urgency"], u)]
        if maybe(0.7): frags.append(pick(FR["budget"], b))
        if maybe(0.7): frags.append(pick(FR["program"], p))
        frags.append(pick(FR["engage"], e))
        if e == 2 and maybe(0.5):
            frags.insert(int(rng.integers(0, len(frags) + 1)), rng.choice(FILLER))
        user_text = ", ".join(str(f) for f in frags)
        # fields (english schema): scored fields keyed to dominant drivers, noisier than text
        fields = {}
        if r >= 1 and p >= 1 and maybe(0.8):
            fields["situation"] = rng.choice(["для работы", "сдать экзамен", "для собеседования", "общение с клиентами"])
        elif maybe(0.3):
            fields["situation"] = "просто для себя"
        if u == 2:   fields["start_date"] = rng.choice(["на этой неделе", "хочу начать сейчас"])
        elif u == 1: fields["start_date"] = "в этом месяце"
        else:        fields["start_date"] = rng.choice(["потом", "пока не знаю", None])
        if r == 2:   fields["hours_per_week"] = str(int(rng.choice([5, 6, 8])))
        elif r == 1: fields["hours_per_week"] = str(int(rng.choice([3, 4])))
        elif maybe(0.7): fields["hours_per_week"] = str(int(rng.choice([1, 2])))
        if r >= 1 and maybe(0.6):
            fields["past_attempts"] = rng.choice(["курсы", "репетитор", "приложение"])
        if maybe(0.7):
            fields["level"] = rng.choice(["начинающий", "средний", "продвинутый"])
        for k in list(fields):
            if maybe(0.12): fields[k] = None
        transcript = [
            {"role": "assistant", "text": "Здравствуйте! Подскажите, для чего сейчас английский?"},
            {"role": "user", "text": user_text},
            {"role": "assistant", "text": "Угу. Когда хотите начать?"},
        ]
        return json.dumps({"transcript": transcript, "fields": fields}, ensure_ascii=False)
    return complete

def load_or_smoke():
    train = _read_jsonl(RAW / "train.jsonl")
    test  = _read_jsonl(RAW / "test.jsonl")
    ood   = _read_jsonl(RAW / "ood.jsonl")
    real  = _read_jsonl(Path("data/real/real.jsonl"))
    if train and test:
        return "REAL (data/raw — LLM-generated)", train, test, ood, real
    rng = np.random.default_rng(RNG_SEED)
    a = generate_dataset([ENG], 1200, rng, make_offline_complete_fn(1, "A"), 0.9, "A")
    rng.shuffle(a)
    n_test = int(len(a) * 0.25)
    test, train = a[:n_test], a[n_test:]
    ood = generate_dataset([ENG], 300, rng, make_offline_complete_fn(2, "B"), 1.1, "B")
    return "OFFLINE-SMOKE (synthetic stub — NOT LLM, NOT real)", train, test, ood, []

DATA_MODE, train, test, ood, real = load_or_smoke()
print("DATA MODE:", DATA_MODE)
print(f"train={len(train)} test={len(test)} ood={len(ood)} real={len(real)}")

## 2. Class balance & provenance

In [ ]:
def pos_rate(recs): return float(np.mean([r["label"] for r in recs])) if recs else float("nan")
splits = {"train": train, "test": test, "ood": ood}
if real: splits["real"] = real
summary = pd.DataFrame({"n": {k: len(v) for k, v in splits.items()},
                        "pos_rate": {k: round(pos_rate(v), 3) for k, v in splits.items()}})
display(summary)
ax = summary["pos_rate"].plot(kind="bar", color="#4C72B0")
ax.axhline(0.5, ls="--", c="gray"); ax.set_ylabel("P(enrolled=1)"); ax.set_ylim(0, 1)
ax.set_title(f"Class balance by split  [{DATA_MODE}]")
plt.tight_layout(); plt.show()

## 3. Distributions by label — transcript length & field completeness

If positives and negatives differ in these *structural* features, the model has
non-text signal to use too. (Computed on the held-out **test** split.)

In [ ]:
def feat_frame(recs):
    rows = []
    for r in recs:
        f = structured_features(r)
        rows.append({"label": r["label"], "completeness": f[0], "n_turns": f[1],
                     "user_turns": f[2], "text_len": f[3], "avg_user_len": f[4]})
    return pd.DataFrame(rows)

fr = feat_frame(test)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lab, grp in fr.groupby("label"):
    axes[0].hist(grp["text_len"], bins=20, alpha=0.6, label=f"label={lab}")
    axes[1].hist(grp["completeness"], bins=10, alpha=0.6, label=f"label={lab}")
axes[0].set_title("Transcript length by label"); axes[0].set_xlabel("chars"); axes[0].legend()
axes[1].set_title("Field completeness by label"); axes[1].set_xlabel("fraction of fields filled"); axes[1].legend()
plt.tight_layout(); plt.show()
display(fr.groupby("label")[["completeness", "text_len", "user_turns"]].mean().round(2))

## 4. Rule baseline — does its `fit` score separate the labels?

`ml.baseline.score_lead` is the faithful Python port of the prod `lib/scoring.js`.
This is the honest, non-strawman competitor (real shipped code).

In [ ]:
bs = baseline_scores(test, SCS_BY_ID)
y = np.array([r["label"] for r in test])
fig, ax = plt.subplots()
ax.boxplot([bs[y == 0], bs[y == 1]])
ax.set_xticklabels(["label=0", "label=1"])
ax.set_ylabel("rule baseline fit (0–100)"); ax.set_title("Rule baseline fit by label (test)")
plt.tight_layout(); plt.show()
print(f"baseline fit  pos mean={bs[y == 1].mean():.1f}   neg mean={bs[y == 0].mean():.1f}")

## 5. The headline — learned model vs the rules

Primary framing is **ranking** (*who to call first*): `precision@20%` and `NDCG@20%`
at a realistic call-back budget. Secondary: `AUC` / `AP` (probability quality).
The model is scored against the **same** records as the rule baseline.

In [ ]:
model, vec = train_model(train)
blocks = {"test": compare(test, model, vec, SCS_BY_ID)}
if ood:  blocks["ood"]  = compare(ood,  model, vec, SCS_BY_ID)
if real: blocks["real"] = compare(real, model, vec, SCS_BY_ID)

rows = []
for split, blk in blocks.items():
    for who in ("model", "baseline"):
        m = blk[who]
        rows.append({"split": split, "scorer": who, "n": blk["n"],
                     "AUC": round(m["auc"], 3), "AP": round(m["ap"], 3),
                     "P@20%": round(m["precision_at_20pct"], 3),
                     "NDCG@20%": round(m["ndcg_at_20pct"], 3)})
results = pd.DataFrame(rows)
display(results)

piv = results.pivot(index="split", columns="scorer", values="P@20%")
ax = piv.plot(kind="bar", color={"model": "#55A868", "baseline": "#C44E52"})
ax.set_ylabel("precision@20%"); ax.set_ylim(0, 1)
ax.set_title("Who-to-call-first: model vs rules (precision@20%)")
plt.tight_layout(); plt.show()

## 6. Leakage guard + the honest ceiling

**Leakage guard:** a deliberately dumb bag-of-words model under 5-fold CV. If it
approached ~1.0, the label leaked into obvious tokens. We want it **clearly below 1.0**.

**Ceiling:** the label is `sigmoid(w·θ − b + ε)` over *continuous* θ with noise ε,
but every observable (text, fields) encodes only **3-level bands** of θ. So even an
oracle is capped — modest AUCs (~0.6) are the *expected, honest* outcome, not a bug.
This irreducible error is deliberate (spec §2).

In [ ]:
lk = leakage_auc(train)
print(f"leakage_auc (dumb bag-of-words, 5-fold CV) = {lk:.3f}   (want clearly < 1.0)")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from ml.personas import LATENT_TRAITS, _band

ALL = train + test
if all("theta" in r for r in ALL):  # theta exists for synthetic data only
    yb = np.array([r["label"] for r in ALL])
    Xc = np.array([[r["theta"][t] for t in LATENT_TRAITS] for r in ALL])
    Xb = np.array([[_band(r["theta"][t]) for t in LATENT_TRAITS] for r in ALL], dtype=float)
    pc = cross_val_predict(LogisticRegression(max_iter=1000), Xc, yb, cv=5, method="predict_proba")[:, 1]
    pb = cross_val_predict(LogisticRegression(max_iter=1000), Xb, yb, cv=5, method="predict_proba")[:, 1]
    print(f"continuous-θ oracle AUC     = {evaluate_scores(yb, pc)['auc']:.3f}   (upper bound, exact θ)")
    print(f"band-observable ceiling AUC = {evaluate_scores(yb, pb)['auc']:.3f}   (max from 3-level bands)")
else:
    print("(θ not present — real data — ceiling diagnostic skipped)")

## 7. Calibration — is the score a usable probability?

The callback priority should be a *probability*, not just a rank. Reliability curve
+ Brier score on the test split.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

ps = model_scores(test, model, vec)
y = np.array([r["label"] for r in test])
frac_pos, mean_pred = calibration_curve(y, ps, n_bins=8, strategy="quantile")
fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], "--", c="gray", label="perfectly calibrated")
ax.plot(mean_pred, frac_pos, "o-", c="#4C72B0", label="model")
ax.set_xlabel("mean predicted P(enrolled)"); ax.set_ylabel("empirical frequency")
ax.set_title(f"Reliability curve   (Brier = {brier_score_loss(y, ps):.3f})"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Limitations & the honest data story

- **Labels are synthetic** (`enrolled` derived from latent θ + noise). Real
  enrollment outcomes can't be observed without live conversions (out of scope).
- **This run is OFFLINE-SMOKE** unless `data/raw/` holds LLM-generated calls. The
  stub is doubly-synthetic (not even an LLM) — every number above is **illustrative
  scaffolding** so the pipeline executes. It is **not** a result.
- **No real-call set here.** The honest sim-to-real bridge is ~20–50 *genuinely
  collected* calls (author + friends, via the prod app, hand-labeled), reported
  with their **true (small) N**. Collect them, drop `data/real/real.jsonl`, and the
  `real` row appears above. We never present synthetic calls as real.
- **Generator dependence.** The OOD (gen-B) split tests whether the signal survives
  a surface/style shift; the leakage guard tests whether it's a token artifact.
  Both are necessary precisely because we bootstrap from generated data — *that
  scrutiny is the project's thesis.*

### To produce the real study
```bash
cp .env.example .env        # set GEN_PROVIDER, GEN_MODEL, and the matching API key
set -a; source .env; set +a
make data                   # LLM-generated gen-A + gen-B  ->  data/raw/
make train                  # logistic-regression scorer   ->  models/
make eval                   # model vs baseline + leakage/OOD/real -> docs/report/metrics.json
make report                 # re-render this notebook against the real data
```
The **product/engineering** exhibit (the FirstContact voice agent + CRM) is the
separate prod repo; this repo links to it as the MVP behind the impact estimate.